# 🚆 RAIL — AI-Powered Automatic Block Planning

## Final SIH Prototype Notebook

This notebook combines the original Phase 1, Phase 2 and Phase 3 work into one reproducible analysis and optimization pipeline.

### Pipeline
1. Load and analyse synthetic railway maintenance, asset, train and corridor data
2. Calculate maintenance priority scores
3. Detect train-free maintenance windows
4. Generate a greedy baseline schedule
5. Optimize task-to-window assignments with OR-Tools CP-SAT
6. Compare baseline and optimized planning results
7. Save outputs consumed by the FastAPI backend and React dashboard

> **Important:** All datasets are synthetic prototype/demo data. This system is not connected to or authorized to control real Indian Railways operations.


# 🚆 AI Railway Automatic Block Planning — Phase 1

## Goal
Is notebook ka first goal **AI banana nahi** hai.

Hum pehle samjhenge:
- maintenance data mein kya hai
- critical/overdue tasks kitne hain
- kaunse sections mein zyada maintenance hai
- train traffic kaisa hai
- corridor mein available windows kya hain
- maintenance aur train schedule ke conflicts kaise detect honge

> Ye synthetic/demo data hai. Ye official Indian Railways operational data nahi hai.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("../data")
print("Data folder:", DATA_DIR.resolve())


## 1. Load all datasets

In [ ]:
maintenance = pd.read_csv(DATA_DIR / "maintenance_tasks.csv")
trains = pd.read_csv(DATA_DIR / "train_schedule.csv")
corridor = pd.read_csv(DATA_DIR / "corridor_availability.csv")
goods = pd.read_csv(DATA_DIR / "goods_forecast.csv")
assets = pd.read_csv(DATA_DIR / "assets.csv")
sections = pd.read_csv(DATA_DIR / "sections.csv")

print("Maintenance:", maintenance.shape)
print("Trains:", trains.shape)
print("Corridor:", corridor.shape)
print("Goods forecast:", goods.shape)
print("Assets:", assets.shape)
print("Sections:", sections.shape)


## 2. Inspect maintenance data

In [ ]:
display(maintenance.head(10))
print("\nDepartments:")
print(maintenance["department"].value_counts())

print("\nStatus:")
print(maintenance["status"].value_counts())

print("\nSeverity:")
print(maintenance["severity"].value_counts())


## 3. Find critical and overdue maintenance

In [ ]:
critical_tasks = maintenance[maintenance["severity"] == "Critical"]
overdue_tasks = maintenance[maintenance["status"] == "Overdue"]

print("Critical tasks:", len(critical_tasks))
print("Overdue tasks:", len(overdue_tasks))

display(
    maintenance[
        (maintenance["severity"].isin(["Critical", "High"])) |
        (maintenance["status"] == "Overdue")
    ].head(20)
)


## 4. Maintenance load by section

In [ ]:
section_load = (
    maintenance.groupby("section")
    .agg(
        total_tasks=("task_id", "count"),
        critical_tasks=("severity", lambda x: (x == "Critical").sum()),
        total_hours=("estimated_duration_hours", "sum")
    )
    .sort_values("total_tasks", ascending=False)
)
display(section_load.head(10))


In [ ]:
section_load["total_tasks"].head(10).plot(kind="bar", figsize=(10,5))
plt.title("Top Sections by Maintenance Task Count")
plt.xlabel("Section")
plt.ylabel("Number of Tasks")
plt.tight_layout()
plt.show()


## 5. Maintenance by department

In [ ]:
dept_summary = (
    maintenance.groupby("department")
    .agg(
        tasks=("task_id", "count"),
        maintenance_hours=("estimated_duration_hours", "sum"),
        critical=("severity", lambda x: (x == "Critical").sum()),
        overdue=("status", lambda x: (x == "Overdue").sum())
    )
)
display(dept_summary)


## 6. Train traffic by section

In [ ]:
train_load = (
    trains.groupby(["section", "train_type"])
    .size()
    .unstack(fill_value=0)
)
display(train_load.head(10))


## 7. Convert time to minutes

Scheduling ke liye `10:30` ko minutes-from-midnight mein convert karna easy hota hai.


In [ ]:
def time_to_minutes(t):
    h, m = map(int, t.split(":"))
    return h * 60 + m

trains["arrival_min"] = trains["arrival_time"].apply(time_to_minutes)
trains["departure_min"] = trains["departure_time"].apply(time_to_minutes)
corridor["start_min"] = corridor["start_time"].apply(time_to_minutes)
corridor["end_min"] = corridor["end_time"].apply(time_to_minutes)

display(trains.head())
display(corridor.head())


## 8. Basic train/block conflict detection

In [ ]:
def has_conflict(block_start, block_end, train_start, train_end):
    return max(block_start, train_start) < min(block_end, train_end)

block_start = time_to_minutes("10:00")
block_end = time_to_minutes("12:00")
train_start = time_to_minutes("10:30")
train_end = time_to_minutes("10:40")

print("Conflict:", has_conflict(
    block_start, block_end, train_start, train_end
))


## 9. Find train-free portions of available windows

In [ ]:
def free_windows_for_section(section_id, date_value):
    windows = corridor[
        (corridor["section"] == section_id) &
        (corridor["date"] == date_value) &
        (corridor["availability"] == "Available")
    ][["start_min", "end_min"]].values.tolist()

    section_trains = trains[
        (trains["section"] == section_id) &
        (trains["date"] == date_value)
    ][["arrival_min", "departure_min"]].values.tolist()

    free = []

    for ws, we in windows:
        current = ws

        for ts, te in sorted(section_trains):
            if te <= current or ts >= we:
                continue

            if ts > current:
                free.append((current, min(ts, we)))

            current = max(current, te)

            if current >= we:
                break

        if current < we:
            free.append((current, we))

    return free

print(
    "Free windows:",
    free_windows_for_section("S01", "2026-09-07")
)


## 10. Basic priority scoring preview

Next phase mein isko better normalize karke full priority engine banayenge.


In [ ]:
severity_score = {
    "Critical": 10, "High": 7, "Medium": 5, "Low": 2
}
safety_score = {"Yes": 10, "No": 3}
importance_score = {"High": 10, "Medium": 6, "Low": 3}

priority_preview = maintenance.copy()
priority_preview["severity_score"] = priority_preview["severity"].map(severity_score)
priority_preview["safety_score"] = priority_preview["safety_critical"].map(safety_score)

priority_preview = priority_preview.merge(
    assets[["asset_id", "importance"]],
    on="asset_id",
    how="left"
)

priority_preview["asset_importance_score"] = (
    priority_preview["importance"].map(importance_score)
)

priority_preview["basic_priority_score"] = (
    priority_preview["severity_score"] * 0.40 +
    priority_preview["safety_score"] * 0.25 +
    priority_preview["asset_importance_score"] * 0.20 +
    priority_preview["status"].eq("Overdue").astype(int) * 1.5
)

display(
    priority_preview[
        ["task_id", "department", "section", "severity",
         "safety_critical", "status", "importance",
         "basic_priority_score"]
    ].sort_values("basic_priority_score", ascending=False).head(20)
)


# 🚆 Phase 2 — Priority Engine + Conflict-Free Block Scheduler

Phase 1 mein data analysis hua. Ab hum actual planning engine banayenge.

Flow:

**Maintenance + Asset Data**
→ **Priority Score**
→ **Train Conflict Detection**
→ **Available Corridor Windows**
→ **Block Assignment**
→ **Multi-Department Grouping**

> Ye synthetic prototype hai, official Indian Railways operational software nahi.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().parent
if not (ROOT/"data"/"maintenance_tasks.csv").exists():
    ROOT = Path("..").resolve()

DATA = ROOT/"data"
OUT = ROOT/"outputs"
OUT.mkdir(exist_ok=True)

maintenance = pd.read_csv(DATA/"maintenance_tasks.csv")
trains = pd.read_csv(DATA/"train_schedule.csv")
corridor = pd.read_csv(DATA/"corridor_availability.csv")
assets = pd.read_csv(DATA/"assets.csv")

print("Maintenance:", maintenance.shape)
print("Trains:", trains.shape)
print("Corridor:", corridor.shape)
print("Assets:", assets.shape)


## 1. Priority Engine

Priority factors:

- Severity
- Safety criticality
- Asset importance
- Due-date urgency
- Overdue status

Highest score = task ko pehle schedule karna hai.


In [ ]:
START=pd.Timestamp("2026-09-07")
SEV={"Critical":10,"High":7,"Medium":5,"Low":2}
SAFE={"Yes":10,"No":3}
IMP={"High":10,"Medium":6,"Low":3}

tasks=maintenance[maintenance.status.isin(["Pending","Overdue"])].copy()
tasks["due_date"]=pd.to_datetime(tasks["due_date"])
tasks["days_until_due"]=(tasks["due_date"]-START).dt.days
tasks["urgency_score"]=np.clip(10-tasks["days_until_due"].clip(-10,10),0,20)
tasks["severity_score"]=tasks.severity.map(SEV)
tasks["safety_score"]=tasks.safety_critical.map(SAFE)
tasks=tasks.merge(assets[["asset_id","importance"]],on="asset_id",how="left")
tasks["asset_importance_score"]=tasks.importance.map(IMP)

tasks["priority_score"]=(
    tasks.severity_score*3.5+
    tasks.safety_score*2.5+
    tasks.asset_importance_score*1.5+
    tasks.urgency_score*1.5+
    np.where(tasks.status.eq("Overdue"),10,0)
)
tasks=tasks.sort_values("priority_score",ascending=False).reset_index(drop=True)

display(tasks[["task_id","department","section","severity","status","importance","days_until_due","priority_score"]].head(20))


In [ ]:
tasks.priority_score.plot(kind="hist",bins=15,figsize=(9,5))
plt.title("Maintenance Priority Score Distribution")
plt.xlabel("Priority Score")
plt.ylabel("Number of Tasks")
plt.tight_layout()
plt.show()


## 2. Train and corridor time conversion

In [ ]:
def tm(x):
    h,m=map(int,x.split(":")); return h*60+m
def ts(x):
    x=int(round(x)); return f"{x//60:02d}:{x%60:02d}"

trains["arrival_min"]=trains.arrival_time.map(tm)
trains["departure_min"]=trains.departure_time.map(tm)
corridor["start_min"]=corridor.start_time.map(tm)
corridor["end_min"]=corridor.end_time.map(tm)


## 3. Train-free maintenance windows

In [ ]:
def train_intervals(section,date):
    x=trains[(trains.section==section)&(trains.date==date)]
    return list(x[["arrival_min","departure_min"]].itertuples(index=False,name=None))

def free_intervals(section,date):
    wins=corridor[(corridor.section==section)&(corridor.date==date)&(corridor.availability=="Available")]
    free=[]
    for w in wins.itertuples(index=False):
        s,e=int(w.start_min),int(w.end_min); cur=s
        for a,b in sorted(train_intervals(section,date)):
            if b<=cur: continue
            if a>=e: break
            if a>cur: free.append((cur,min(a,e)))
            cur=max(cur,b)
            if cur>=e: break
        if cur<e: free.append((cur,e))
    return free

print("S01 / 2026-09-07:")
print([(ts(a),ts(b)) for a,b in free_intervals("S01","2026-09-07")])


## 4. Greedy Block Scheduler

Rules:

1. Highest-priority tasks first.
2. Only Pending/Overdue tasks.
3. Block must fit inside an available corridor window.
4. Block cannot overlap a train.
5. Same section/date tasks are grouped when possible.
6. Grouping multiple departments demonstrates coordinated planning.


In [ ]:
def overlap(a,b,c,d):
    return max(a,c)<min(b,d)

def schedule(tasks):
    dates=sorted(set(corridor.date)&set(pd.date_range("2026-09-07","2026-09-20").strftime("%Y-%m-%d")))
    blocks=[]; results=[]

    for _,task in tasks.iterrows():
        dur=float(task.estimated_duration_hours)*60
        placed=False

        for date in dates:
            for fs,fe in free_intervals(task.section,date):
                existing=None
                for b in blocks:
                    if (b["section"]==task.section and b["date"]==date and
                        b["block_end"]>=fs and b["block_end"]+dur<=fe):
                        existing=b; break

                if existing:
                    new_end=existing["block_end"]+dur
                    if any(overlap(existing["block_start"],new_end,a,b)
                           for a,b in train_intervals(task.section,date)):
                        continue
                    existing["block_end"]=new_end
                    existing["task_ids"].append(task.task_id)
                    existing["departments"].add(task.department)
                    existing["maintenance_hours"]+=float(task.estimated_duration_hours)
                    results.append([task.task_id,task.department,task.section,round(task.priority_score,2),
                                    existing["block_id"],date,ts(existing["block_start"]),ts(new_end),
                                    "Scheduled","Grouped into existing block"])
                    placed=True; break

                if fe-fs>=dur:
                    bid=f"B{len(blocks)+1:03d}"
                    blocks.append({"block_id":bid,"section":task.section,"date":date,
                                   "block_start":fs,"block_end":fs+dur,"window_end":fe,
                                   "task_ids":[task.task_id],"departments":{task.department},
                                   "maintenance_hours":float(task.estimated_duration_hours)})
                    results.append([task.task_id,task.department,task.section,round(task.priority_score,2),
                                    bid,date,ts(fs),ts(fs+dur),"Scheduled","New conflict-free block"])
                    placed=True; break
            if placed: break

        if not placed:
            results.append([task.task_id,task.department,task.section,round(task.priority_score,2),
                            "","","","","Unscheduled","No suitable conflict-free window"])

    task_plan=pd.DataFrame(results,columns=["task_id","department","section","priority_score","block_id",
                                            "date","block_start","block_end","status","reason"])
    rows=[]
    for b in blocks:
        duration=b["block_end"]-b["block_start"]
        util=(b["maintenance_hours"]*60/duration*100) if duration else 0
        rows.append([b["block_id"],b["date"],b["section"],ts(b["block_start"]),ts(b["block_end"]),
                     round(duration/60,2),round(b["maintenance_hours"],2),round(min(util,100),2),
                     len(b["task_ids"]),", ".join(sorted(b["departments"])),", ".join(b["task_ids"])])
    block_plan=pd.DataFrame(rows,columns=["block_id","date","section","block_start","block_end",
                                           "block_duration_hours","maintenance_hours","block_utilization_pct",
                                           "task_count","departments","task_ids"])
    return task_plan,block_plan

task_plan,block_plan=schedule(tasks)
display(block_plan.head(20))


## 5. Evaluate the result

In [ ]:
scheduled=(task_plan.status=="Scheduled").sum()
unscheduled=(task_plan.status=="Unscheduled").sum()
multi=(block_plan.departments.str.contains(",",regex=False).sum() if len(block_plan) else 0)

print("Total tasks considered :",len(task_plan))
print("Scheduled              :",scheduled)
print("Unscheduled            :",unscheduled)
print("Blocks created         :",len(block_plan))
print("Multi-department blocks:",multi)
if len(block_plan):
    print("Average utilization    :",round(block_plan.block_utilization_pct.mean(),2),"%")


In [ ]:
multi_blocks=block_plan[block_plan.departments.str.contains(",",regex=False)] if len(block_plan) else block_plan
display(multi_blocks)


## 6. Save Phase 2 outputs

In [ ]:
tasks.to_csv(OUT/"priority_tasks.csv",index=False)
task_plan.to_csv(OUT/"optimized_task_assignments.csv",index=False)
block_plan.to_csv(OUT/"optimized_block_plan.csv",index=False)

print("✓ priority_tasks.csv")
print("✓ optimized_task_assignments.csv")
print("✓ optimized_block_plan.csv")


# ✅ Phase 2 Complete

Ab tumhare prototype mein working hai:

- Priority Engine
- Urgency calculation
- Train conflict detection
- Corridor availability checking
- Conflict-free scheduling
- Same-section multi-department grouping
- Optimized CSV output

### Phase 3
Ab isi problem ko **OR-Tools optimization** se solve karenge, jahan objective hoga:

**Minimize block count + maintenance delay + train disruption**
and
**Maximize asset availability + block utilization + coordinated maintenance.**


# 🚆 Phase 3 — Advanced Block Optimization using OR-Tools

Phase 2 used a priority-based greedy scheduler. Phase 3 upgrades it to an optimization model.

The model considers maintenance priority, corridor availability, train conflicts, task duration, block capacity, and the number of separate blocks.

> This is an SIH prototype using synthetic/demo data, not a live railway control system.


In [ ]:
# Cell 1 — Imports and project paths
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ortools.sat.python import cp_model

ROOT = Path.cwd().parent
if not (ROOT / "data" / "maintenance_tasks.csv").exists():
    ROOT = Path("..").resolve()

DATA = ROOT / "data"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

print("Project root:", ROOT)
print("OR-Tools imported successfully.")


In [ ]:
# Cell 2 — Load data
maintenance = pd.read_csv(DATA / "maintenance_tasks.csv")
trains = pd.read_csv(DATA / "train_schedule.csv")
corridor = pd.read_csv(DATA / "corridor_availability.csv")
assets = pd.read_csv(DATA / "assets.csv")

priority_file = OUT / "priority_tasks.csv"

if priority_file.exists():
    tasks = pd.read_csv(priority_file)
    print("Using Phase 2 priority_tasks.csv")
else:
    tasks = maintenance[maintenance["status"].isin(["Pending", "Overdue"])].copy()
    sev = {"Critical":10, "High":7, "Medium":5, "Low":2}
    safe = {"Yes":10, "No":3}
    imp = {"High":10, "Medium":6, "Low":3}
    tasks["due_date"] = pd.to_datetime(tasks["due_date"])
    ref = pd.Timestamp("2026-09-07")
    days = (tasks["due_date"] - ref).dt.days
    tasks["urgency_score"] = np.clip(10 - days.clip(-10,10), 0, 20)
    tasks["severity_score"] = tasks["severity"].map(sev).fillna(1)
    tasks["safety_score"] = tasks["safety_critical"].map(safe).fillna(1)
    tasks = tasks.merge(assets[["asset_id","importance"]], on="asset_id", how="left")
    tasks["asset_importance_score"] = tasks["importance"].map(imp).fillna(3)
    tasks["priority_score"] = (
        tasks["severity_score"]*3.5 +
        tasks["safety_score"]*2.5 +
        tasks["asset_importance_score"]*1.5 +
        tasks["urgency_score"]*1.5 +
        np.where(tasks["status"].eq("Overdue"), 10, 0)
    )

tasks["due_date"] = pd.to_datetime(tasks["due_date"])
print("Tasks:", len(tasks))
display(tasks.head())


## 1. Convert time fields to minutes

In [ ]:
# Cell 3 — Time conversion
def to_minute(value):
    h, m = map(int, str(value).split(":")[:2])
    return h*60 + m

def to_time(value):
    value = int(round(value))
    return f"{value//60:02d}:{value%60:02d}"

trains["arrival_min"] = trains["arrival_time"].map(to_minute)
trains["departure_min"] = trains["departure_time"].map(to_minute)
corridor["start_min"] = corridor["start_time"].map(to_minute)
corridor["end_min"] = corridor["end_time"].map(to_minute)
print("Time conversion complete.")


## 2. Generate train-free corridor windows

Available corridor periods are split around train movements, so maintenance can only use train-free time.


In [ ]:
# Cell 4 — Generate train-free windows
def train_intervals(section, date):
    x = trains[(trains["section"] == section) & (trains["date"] == date)]
    return sorted(list(x[["arrival_min","departure_min"]].itertuples(index=False, name=None)))

def make_free_windows(section, date):
    available = corridor[
        (corridor["section"] == section) &
        (corridor["date"] == date) &
        (corridor["availability"] == "Available")
    ]
    result = []
    for row in available.itertuples(index=False):
        start, end = int(row.start_min), int(row.end_min)
        current = start
        for ts, te in train_intervals(section, date):
            if te <= current:
                continue
            if ts >= end:
                break
            if ts > current:
                result.append((current, min(ts, end)))
            current = max(current, te)
            if current >= end:
                break
        if current < end:
            result.append((current, end))
    return [(s,e) for s,e in result if e > s]

rows = []
for section in sorted(corridor["section"].unique()):
    for date in sorted(corridor["date"].unique()):
        for start, end in make_free_windows(section, date):
            rows.append({
                "window_id": f"W{len(rows)+1:04d}",
                "section": section, "date": date,
                "start_min": start, "end_min": end,
                "capacity_min": end-start
            })

windows = pd.DataFrame(rows)
print("Train-free windows:", len(windows))
display(windows.head(15))


## 3. Create task → window candidates

In [ ]:
# Cell 5 — Candidate assignments
candidate_rows = []

for task_index, task in tasks.iterrows():
    duration = int(round(float(task["estimated_duration_hours"]) * 60))
    possible = windows[
        (windows["section"] == task["section"]) &
        (windows["capacity_min"] >= duration)
    ]
    for window_index, window in possible.iterrows():
        candidate_rows.append({
            "task_index": task_index,
            "task_id": task["task_id"],
            "window_index": window_index,
            "window_id": window["window_id"],
            "section": window["section"],
            "date": window["date"],
            "duration_min": duration,
            "capacity_min": int(window["capacity_min"]),
            "priority": float(task["priority_score"]),
            "due_date": task["due_date"]
        })

candidates = pd.DataFrame(candidate_rows)
print("Candidate assignments:", len(candidates))
display(candidates.head(15))


## 4. Build the CP-SAT optimization model

Each candidate has a binary decision variable. A task can be selected at most once, and each maintenance window has limited capacity.

The objective rewards high-priority tasks and penalizes delayed work and excessive separate windows.


In [ ]:
# Cell 6 — CP-SAT model
model = cp_model.CpModel()

x = {i: model.NewBoolVar(f"x_{i}") for i in candidates.index}

# A task can be scheduled at most once.
for task_index in tasks.index:
    ids = candidates.index[candidates["task_index"] == task_index].tolist()
    if ids:
        model.Add(sum(x[i] for i in ids) <= 1)

# Window capacity constraints.
for window_index in windows.index:
    ids = candidates.index[candidates["window_index"] == window_index].tolist()
    if ids:
        capacity = int(windows.loc[window_index, "capacity_min"])
        model.Add(
            sum(int(candidates.loc[i, "duration_min"]) * x[i] for i in ids)
            <= capacity
        )

# Window-used variables.
window_used = {}
for wi in windows.index:
    window_used[wi] = model.NewBoolVar(f"window_used_{wi}")
    ids = candidates.index[candidates["window_index"] == wi].tolist()
    for i in ids:
        model.Add(x[i] <= window_used[wi])

# Objective.
objective = []
for i in candidates.index:
    priority = float(candidates.loc[i, "priority"])
    due = pd.Timestamp(candidates.loc[i, "due_date"])
    wdate = pd.Timestamp(candidates.loc[i, "date"])
    delay_days = max(0, (wdate - due).days)

    reward = int(round(priority * 100))
    delay_penalty = delay_days * 120
    small_time_penalty = int(candidates.loc[i, "window_index"]) % 100

    objective.append((reward - delay_penalty - small_time_penalty) * x[i])

BLOCK_PENALTY = 180
for wi in windows.index:
    objective.append(-BLOCK_PENALTY * window_used[wi])

model.Maximize(sum(objective))

solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30
solver.parameters.num_search_workers = 8

status = solver.Solve(model)
print("Solver status:", solver.StatusName(status))
print("Objective value:", solver.ObjectiveValue())


## 5. Extract optimized assignments

In [ ]:
# Cell 7 — Extract solution
selected = [candidates.loc[i].to_dict() for i in candidates.index if solver.Value(x[i]) == 1]
optimized_assignments = pd.DataFrame(selected)

if len(optimized_assignments):
    optimized_assignments["block_start"] = optimized_assignments.apply(
        lambda r: int(windows.loc[int(r["window_index"]), "start_min"]), axis=1
    )
    optimized_assignments["block_end"] = (
        optimized_assignments["block_start"] + optimized_assignments["duration_min"]
    )
    optimized_assignments["block_start_time"] = optimized_assignments["block_start"].map(to_time)
    optimized_assignments["block_end_time"] = optimized_assignments["block_end"].map(to_time)
    optimized_assignments = optimized_assignments.merge(
        tasks[["task_id","department","asset_id","severity","status"]],
        on="task_id", how="left"
    )

print("Scheduled tasks:", len(optimized_assignments))
print("Unscheduled tasks:", len(tasks) - len(optimized_assignments))
display(optimized_assignments.head(20))


## 6. Build coordinated optimized blocks

Assignments using the same section/date/window are grouped into one coordinated block. This supports Engineering + S&T + Traction working together.


In [ ]:
# Cell 8 — Build optimized blocks
if len(optimized_assignments):
    block_rows = []
    for (window_id, section, date), group in optimized_assignments.groupby(
        ["window_id","section","date"]
    ):
        window = windows[windows["window_id"] == window_id].iloc[0]
        start = int(window["start_min"])
        capacity = int(window["capacity_min"])
        used = int(group["duration_min"].sum())
        end = start + used
        departments = sorted(group["department"].dropna().unique().tolist())
        task_ids = group["task_id"].astype(str).tolist()

        block_rows.append({
            "block_id": f"OPT-{window_id}",
            "window_id": window_id,
            "date": date,
            "section": section,
            "block_start": to_time(start),
            "block_end": to_time(end),
            "block_duration_hours": round(used/60, 2),
            "available_capacity_hours": round(capacity/60, 2),
            "utilization_pct": round(used/capacity*100, 2) if capacity else 0,
            "task_count": len(task_ids),
            "departments": ", ".join(departments),
            "task_ids": ", ".join(task_ids)
        })
    optimized_blocks = pd.DataFrame(block_rows)
else:
    optimized_blocks = pd.DataFrame(columns=[
        "block_id","window_id","date","section","block_start","block_end",
        "block_duration_hours","available_capacity_hours","utilization_pct",
        "task_count","departments","task_ids"
    ])

display(optimized_blocks.head(20))


## 7. Phase 3 KPIs

In [ ]:
# Cell 9 — KPIs
total_tasks = len(tasks)
scheduled = len(optimized_assignments)
completion_rate = scheduled / total_tasks * 100 if total_tasks else 0
block_count = len(optimized_blocks)
avg_utilization = optimized_blocks["utilization_pct"].mean() if block_count else 0
multi_department = (
    optimized_blocks["departments"].str.contains(",", regex=False).sum()
    if block_count else 0
)
critical_total = int((tasks["severity"] == "Critical").sum())
critical_scheduled = (
    int((optimized_assignments["severity"] == "Critical").sum()) if scheduled else 0
)
overdue_total = int((tasks["status"] == "Overdue").sum())
overdue_scheduled = (
    int((optimized_assignments["status"] == "Overdue").sum()) if scheduled else 0
)

print("========== PHASE 3 KPI ==========")
print(f"Total tasks               : {total_tasks}")
print(f"Scheduled tasks           : {scheduled}")
print(f"Completion rate           : {completion_rate:.2f}%")
print(f"Optimized blocks          : {block_count}")
print(f"Average utilization      : {avg_utilization:.2f}%")
print(f"Multi-department blocks  : {multi_department}")
print(f"Critical scheduled       : {critical_scheduled}/{critical_total}")
print(f"Overdue scheduled        : {overdue_scheduled}/{overdue_total}")


## 8. Visualize optimized block utilization

In [ ]:
# Cell 10 — Utilization chart
if len(optimized_blocks):
    plot_data = optimized_blocks.head(20)
    plt.figure(figsize=(11,5))
    plt.bar(plot_data["block_id"], plot_data["utilization_pct"])
    plt.title("Phase 3 — Optimized Block Utilization")
    plt.xlabel("Optimized Block")
    plt.ylabel("Utilization (%)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No optimized blocks available.")


## 9. Compare Phase 2 and Phase 3

In [ ]:
# Cell 11 — Phase 2 vs Phase 3
p2_blocks_file = OUT / "optimized_block_plan.csv"
p2_tasks_file = OUT / "optimized_task_assignments.csv"

if p2_blocks_file.exists():
    p2 = pd.read_csv(p2_blocks_file)

    if "block_utilization_pct" in p2.columns:
        p2_util = p2["block_utilization_pct"].mean()
    elif "utilization_pct" in p2.columns:
        p2_util = p2["utilization_pct"].mean()
    else:
        p2_util = np.nan

    if p2_tasks_file.exists():
        p2_tasks = pd.read_csv(p2_tasks_file)
        p2_scheduled = (
            int(p2_tasks["status"].astype(str).str.lower().eq("scheduled").sum())
            if "status" in p2_tasks.columns else len(p2_tasks)
        )
    else:
        p2_scheduled = np.nan

    comparison = pd.DataFrame({
        "Metric": ["Scheduled tasks","Number of blocks","Average block utilization (%)"],
        "Phase 2 — Greedy": [
            p2_scheduled, len(p2),
            round(p2_util,2) if pd.notna(p2_util) else np.nan
        ],
        "Phase 3 — OR-Tools": [
            scheduled, block_count, round(avg_utilization,2)
        ]
    })
    display(comparison)
else:
    print("Phase 2 block output not found.")


## 10. Save Phase 3 outputs

Phase 2 files are preserved and are not overwritten.

In [ ]:
# Cell 12 — Save outputs
optimized_assignments.to_csv(
    OUT / "phase3_optimized_task_assignments.csv", index=False
)
optimized_blocks.to_csv(
    OUT / "phase3_optimized_block_plan.csv", index=False
)
windows.to_csv(
    OUT / "phase3_feasible_windows.csv", index=False
)

print("✓ phase3_optimized_task_assignments.csv")
print("✓ phase3_optimized_block_plan.csv")
print("✓ phase3_feasible_windows.csv")
print("Phase 2 outputs were NOT overwritten.")


# ✅ Final optimization pipeline complete

The notebook has completed the data-analysis, priority/scheduling and CP-SAT optimization stages.

### Backend-ready outputs
- `outputs/priority_tasks.csv`
- `outputs/optimized_task_assignments.csv`
- `outputs/optimized_block_plan.csv`
- `outputs/phase3_optimized_task_assignments.csv`
- `outputs/phase3_optimized_block_plan.csv`
- `outputs/phase3_feasible_windows.csv`

Start the FastAPI backend after running this notebook, then launch the React dashboard.
